# Results  

Final results from the pipeline are stored in subdirectory *plateanalysis/footprints/results/*. 

Each file contains the result of a pipeline run over a *sequence* of plates. Sequences of plates are defined in file *settings.py*, and created based on the output of notebook *footprints.ipynb*.

The result file contains just the very final product; intermediate results are not stored in GitHub due to the policy of not storing there files that are either temporary, or potentially large, such as FITS binary tables and such. Intermediate results can be seen only by running the code locally over the appropriate data sets.  

#### Note

Due to sheer size, only one of the plate sequences I worked with has its pipeline output uploaded to the repo. Also, GitHub is unable to display the html due to its size. Download the file and use your browser locally to display the file.  

Alternatively, you can download the result files from Google Drive: https://drive.google.com/drive/folders/164xc4faMDbPt84ZileJZoHzK94kvG8Gm


Below is a summary of findings to date. Numbers refer to the plate ID code in the APPLAUSE archive.

- Data from the **1m-Spiegelteleskop** and the **Doppel-Reflektor** telescopes: still to be reviewed in light of the upgraded software.

- Data from the **Grosser Schmidt-Spiegel** telescope yeld a few candidates.




### Notes

APPLAUSE scans were performed on the original plates, not copies.

## Processing stats

Summary of number of detections/objects for each plate pair.

In [1]:
import os
import pandas as pd
from astropy.table import Table

import settings
from settings import get_parameters, fname, sequences

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
def make_table():
    table = Table(names=['seq', 'plate 1', 'plate 2', 'total sources', 'matched', 'non-matched', 'candidates'],
                  dtype=['S1', 'S1', 'S1', 'i4', 'i4', 'i4', 'i4'])

    return table

In [3]:
def display_with_pandas(table):
    df = table.to_pandas()

    df['seq'] = df['seq'].apply(lambda x: x.decode('utf-8'))
    df['plate 1'] = df['plate 1'].apply(lambda x: x.decode('utf-8'))
    df['plate 2'] = df['plate 2'].apply(lambda x: x.decode('utf-8'))
    
    display(df)

In [4]:
result = make_table()

for seq_key in list(sequences.keys()):
    seq = sequences[seq_key]

    for i in range(len((seq)) - 1):
        try:
            plate_id_str = str(seq[i])
            next_plate_id_str = str(seq[i+1])

            key = plate_id_str + ',' + next_plate_id_str
            par = get_parameters(key)

            table_sources = Table.read(fname(par['table1']), format='ascii.csv')
            mask = table_sources['scan_id'] == table_sources['scan_id'][0]
            table_sources = table_sources[mask]
            sources = len(table_sources)

            table_matched = Table.read(fname(par['table_matched']), format='fits')
            matched = len(table_matched)

            table_non_matched = Table.read(fname(par['table_non_matched']), format='fits')
            non_matched = len(table_non_matched)

            table_candidates = Table.read(fname(par['table_candidates']), format='fits')
            candidates = len(table_candidates)

            result.add_row([seq_key, plate_id_str, next_plate_id_str, sources, matched, non_matched, candidates])
        except KeyError:
            break

Table columns:

- **seq**: sequence ID generated by notebook *footprints.ipynb*
- **plate ...**: plate IDs defining the pair of plates
- **total sources**: total number of sources in the APPLAUSE table, for the *_x* scan direction
- **matched**: number of sources in the first plate that have a match in the second plate, after the first plate's source table gets prunned of bad detections (as per sextractor-generated parameters)
- **non-matched**: number of sources in the first plate (after it was cleaned up as above) that have **NO** match on the second plate
- **candidates**: number of candidates remaining after filtering is applyied (filtering based on Gaussian fitting, radial profile, and aperture photometry analysis)


In [5]:
display_with_pandas(result)

,seq,plate 1,plate 2,total sources,matched,non-matched,candidates
0,seq00,8794,8795,8444,104,150,3
1,seq00,8795,8796,11754,299,91,5
2,seq01,9245,9246,590001,246737,27726,12
3,seq01,9246,9247,465264,233033,16063,29
4,seq03,9313,9315,55239,17234,93,0
5,seq03,9315,9317,76057,22462,2029,14
6,seq03,9317,9318,59981,21413,426,3
7,seq03,9318,9319,62314,21293,139,3
8,seq03,9319,9320,75926,23992,380,4
9,seq04,9322,9323,58894,20801,315,2


## To do

- add a circularity diagnostic.

- fix *footprints.ipynb*: time order in sequences of plates.

- multivariate probability.

- profile metrics - DONE

- refactor towards batch processing - DONE

- automate data download - DONE

- add Earth shadow angle to tables - DONE

## Summary of possible bright event in plate 9319

- date 1956-12-03
- time between utc_mid: 28 min
- exposure time on each plate: 15 min
- emulsion Kodak Oa-E both plates

Plates 9313, 9315, 9316, 9317, 9318, were searched for similar events, with no detection. These plates were all taken on the same night of 1956-12-03 over a 4 hr time span, with the same telescope pointing (except 9316 that was taken 24 hr later). 

The event position on the plate (distance to center = 3248.3 px, distance to edge = 1061.8 px, annular bin = 4) places it in a region where artifacts can be perhaps more likely to happen. The profile suggests that it might be indeed an artifact: these tend to have systematically narrower widths than stars of same brightness.  

Plate's FOV is far from Earth's shadow. 